## Аналіз A/B-тестів

Ви - аналітик даних в ІТ-компанії і до вас надійшла задача проаналізувати дані A/B тесту в популярній [грі Cookie Cats](https://www.facebook.com/cookiecatsgame). Це - гра-головоломка в стилі «з’єднай три», де гравець повинен з’єднати плитки одного кольору, щоб очистити дошку та виграти рівень. На дошці також зображені співаючі котики :)

Під час проходження гри гравці стикаються з воротами, які змушують їх чекати деякий час, перш ніж вони зможуть прогресувати або зробити покупку в додатку.

У цьому блоці завдань ми проаналізуємо результати A/B тесту, коли перші ворота в Cookie Cats було переміщено з рівня 30 на рівень 40. Зокрема, ми хочемо зрозуміти, як це вплинуло на утримання (retention) гравців. Тобто хочемо зрозуміти, чи переміщення воріт на 10 рівнів пізніше якимось чином вплинуло на те, що користувачі перестають грати в гру раніше чи пізніше з точки зору кількості їх днів з моменту встановлення гри.

Будемо працювати з даними з файлу `cookie_cats.csv`. Колонки в даних наступні:

- `userid` - унікальний номер, який ідентифікує кожного гравця.
- `version` - чи потрапив гравець в контрольну групу (gate_30 - ворота на 30 рівні) чи тестову групу (gate_40 - ворота на 40 рівні).
- `sum_gamerounds` - кількість ігрових раундів, зіграних гравцем протягом першого тижня після встановлення
- `retention_1` - чи через 1 день після встановлення гравець повернувся і почав грати?
- `retention_7` - чи через 7 днів після встановлення гравець повернувся і почав грати?

Коли гравець встановлював гру, його випадковим чином призначали до групи gate_30 або gate_40.

In [2]:
import pandas as pd

data = pd.read_csv('/content/drive/MyDrive/ANALYTICS/Colaboratory/data/cookie_cats.csv')

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


1. Для початку, уявімо, що ми тільки плануємо проведення зазначеного А/B-тесту і хочемо зрозуміти, дані про скількох користувачів нам треба зібрати, аби досягнути відчутного ефекту. Відчутним ефектом ми вважатимемо збільшення утримання на 1% після внесення зміни. Обчисліть, скільки користувачів сумарно нам треба аби досягнути такого ефекту, якщо продакт менеджер нам повідомив, що базове утримання є 19%.

In [29]:
import statsmodels.stats.api as sms

p1 = 0.19
p2 = 0.20

alpha = 0.05
power = 0.80

effect_size = sms.proportion_effectsize(p1, p2)

sample_size_per_group = sms.NormalIndPower().solve_power(
    effect_size=effect_size,
    alpha=alpha,
    power=power,
    alternative='two-sided'
)

n_per_group = int(sample_size_per_group + 0.9999)
total_sample_size = n_per_group * 2

print("\n1. ВХІДНІ ПАРАМЕТРИ ПЛАНУВАННЯ:")
print(f" - Базове утримання (p1):                  {p1*100:.1f}%")
print(f" - Очікуване утримання з ефектом 1% (p2):  {p2*100:.1f}%")
print(f" - Стандартний рівень значущості (alpha):  {alpha}")
print(f" - Стандартна потужність тесту (power):    {power}")

print(f"\n2. РЕЗУЛЬТАТИ РОЗРАХУНКУ РОЗМІРУ ВИБІРКИ:")
print(f" - Необхідно користувачів на ОДНУ групу:   {n_per_group:,} осіб")
print(f" - СУМАРНО користувачів для A/B-тесту:     {total_sample_size:,} осіб")

print(f"\n3. ТЕКСТОВИЙ ВИСНОВОК ДЛЯ ЗДАЧІ:")
print(f" Аби зафіксувати статистично значуще збільшення утримання на 1% (з 19% до 20%) із потужністю 80% та рівнем значущості 5%, нам необхідно зібрати вибірку розміром по {n_per_group:,} користувачів для кожної групи (контрольної та тестової). Таким чином, сумарно нам потрібно зібрати дані про {total_sample_size:,} користувачів.")


1. ВХІДНІ ПАРАМЕТРИ ПЛАНУВАННЯ:
 - Базове утримання (p1):                  19.0%
 - Очікуване утримання з ефектом 1% (p2):  20.0%
 - Стандартний рівень значущості (alpha):  0.05
 - Стандартна потужність тесту (power):    0.8

2. РЕЗУЛЬТАТИ РОЗРАХУНКУ РОЗМІРУ ВИБІРКИ:
 - Необхідно користувачів на ОДНУ групу:   24,638 осіб
 - СУМАРНО користувачів для A/B-тесту:     49,276 осіб

3. ТЕКСТОВИЙ ВИСНОВОК ДЛЯ ЗДАЧІ:
 Аби зафіксувати статистично значуще збільшення утримання на 1% (з 19% до 20%) із потужністю 80% та рівнем значущості 5%, нам необхідно зібрати вибірку розміром по 24,638 користувачів для кожної групи (контрольної та тестової). Таким чином, сумарно нам потрібно зібрати дані про 49,276 користувачів.


2. Зчитайте дані АВ тесту у змінну `df` та виведіть середнє значення показника показник `retention_7` (утримання на 7 день) по версіям гри. Сформулюйте гіпотезу: яка версія дає краще утримання через 7 днів після встановлення гри?

In [30]:
df = pd.read_csv('/content/drive/MyDrive/ANALYTICS/Colaboratory/data/cookie_cats.csv')

retention_by_version = df.groupby('version')['retention_7'].mean()

p_30 = retention_by_version['gate_30']
p_40 = retention_by_version['gate_40']

if p_30 > p_40:
    visual_winner = "gate_30 (ворота на 30 рівні) забезпечує КРАЩЕ утримання користувачів."
else:
    visual_winner = "gate_40 (ворота на 40 рівні) забезпечує КРАЩЕ утримання користувачів."

print(f"\n1. РОЗРАХОВАНІ СЕРЕДНІ ЗНАЧЕННЯ RETENTION_7:")
print(f" - Контрольна група (gate_30): {p_30:.4f} ({p_30*100:.2f}%)")
print(f" - Тестова група    (gate_40): {p_40:.4f} ({p_40*100:.2f}%)")

print("\n2. ПЕРВИННЕ ПРИПУЩЕННЯ (ГІПОТЕЗА):")
print(f" - На основі отриманих середніх значень, версія гри {visual_winner}")

print(f"\n3. СТАТИСТИЧНІ ГІПОТЕЗИ ДЛЯ ПОДАЛЬШОЇ ПЕРЕВІРКИ:")
print(f" - Нульова гіпотеза (H0): p_30 = p_40 (переміщення воріт з 30 на 40 рівень")
print(f"   НЕ вплинуло на утримання гравців на 7-й день).")
print(f" - Альтернативна гіпотеза (Ha): p_30 != p_40 (утримання на 7-й день")
print(f"   статистично значущо ВІДРІЗНЯЄТЬСЯ між двома версіями гри).")



1. РОЗРАХОВАНІ СЕРЕДНІ ЗНАЧЕННЯ RETENTION_7:
 - Контрольна група (gate_30): 0.1902 (19.02%)
 - Тестова група    (gate_40): 0.1820 (18.20%)

2. ПЕРВИННЕ ПРИПУЩЕННЯ (ГІПОТЕЗА):
 - На основі отриманих середніх значень, версія гри gate_30 (ворота на 30 рівні) забезпечує КРАЩЕ утримання користувачів.

3. СТАТИСТИЧНІ ГІПОТЕЗИ ДЛЯ ПОДАЛЬШОЇ ПЕРЕВІРКИ:
 - Нульова гіпотеза (H0): p_30 = p_40 (переміщення воріт з 30 на 40 рівень
   НЕ вплинуло на утримання гравців на 7-й день).
 - Альтернативна гіпотеза (Ha): p_30 != p_40 (утримання на 7-й день
   статистично значущо ВІДРІЗНЯЄТЬСЯ між двома версіями гри).


3. Перевірте з допомогою пасуючого варіанту z-тесту, чи дає якась з версій гри кращий показник `retention_7` на рівні значущості 0.05. Обчисліть також довірчі інтервали для варіантів до переміщення воріт і після. Виведіть результат у форматі:

    ```
    z statistic: ...
    p-value: ...
    Довірчий інтервал 95% для групи control: [..., ...]
    Довірчий інтервал 95% для групи treatment: [..., ...]
    ```

    де замість `...` - обчислені значення.
    
    В якості висновку дайте відповідь на два питання:  

      1. Чи є статистична значущою різниця між поведінкою користувачів у різних версіях гри?   
      2. Чи перетинаються довірчі інтервали утримання користувачів з різних версій гри? Про що це каже?  


In [26]:
import pandas as pd
import numpy as np
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

df = pd.read_csv('/content/drive/MyDrive/ANALYTICS/Colaboratory/data/cookie_cats.csv')

df_control = df[df['version'] == 'gate_30']
success_control = df_control['retention_7'].sum()
n_control = len(df_control)

df_treatment = df[df['version'] == 'gate_40']
success_treatment = df_treatment['retention_7'].sum()
n_treatment = len(df_treatment)

counts = np.array([success_control, success_treatment])
nobs = np.array([n_control, n_treatment])
z_stat, p_value = proportions_ztest(counts, nobs, alternative='two-sided')

ci_control = proportion_confint(success_control, n_control, alpha=0.05, method='normal')
ci_treatment = proportion_confint(success_treatment, n_treatment, alpha=0.05, method='normal')

print(f"z statistic:", round(z_stat, 4))
print(f"p-value:", round(p_value, 5))
print(f"Довірчий інтервал 95% для групи control: [{ci_control[0]:.4f}, {ci_control[1]:.4f}]")
print(f"Довірчий інтервал 95% для групи treatment: [{ci_treatment[0]:.4f}, {ci_treatment[1]:.4f}]")

print(f"\n У якості висновку дайте відповідь на два питання:")

print(f"\n1. Чи є статистична значущою різниця між поведінкою користувачів у різних версіях гри?")
if p_value < 0.05:
    print(f" Відповідь: Так, різниця між версіями гри є статистично значущою.")
    print(f" Обґрунтування: Отримане значення p-value ({p_value:.5f}) є значно меншим за заданий рівень значущості alpha (0.05). Це змушує нас відхилити нульову гіпотезу (H0)")
    print(f" Практичний зміст: Перенесення перших лімітуючих воріт з 30-го на 40-й рівень викликало реальну статистично підтверджену зміну показника retention_7.")
else:
    print(f" Відповідь: Ні, різниця між поведінкою користувачів не є статистично значущою.")
    print(f" Обґрунтування: Оскільки p-value ({p_value:.5f}) >= alpha (0.05), ми не маємо достатніх підстав відхиляти нульову гіпотезу про рівність пропорцій.")

print(f"\n2. Чи перетинаються довірчі інтервали утримання користувачів з різних версій гри? Про що це каже?")

if ci_control[1] < ci_treatment[0] or ci_treatment[1] < ci_control[0]:
    intervals_overlap = False
    print(f" Відповідь: Ні, довірчі інтервали 95% для обох груп не перетинаються взагалі.")
else:
    intervals_overlap = True
    print(f" Відповідь: Так, довірчі інтервали 95% для груп частково або повністю перетинаються.")

print(f" Про що це каже:")
if not intervals_overlap:
    print(f" - Верхня межа інтервалу тестової групи treatment ({ci_treatment[1]:.4f}) повністю знаходиться нижче, ніж нижня межа контрольної групи control ({ci_control[0]:.4f}).")
    print(f" - Це наочно і беззаперечно підтверджує наявність сильного та стабільного ефекту.")
    print(f" - Навіть за найгірших випадкових коливань утримання гравців у версії gate_30 буде вищим, ніж у версії gate_40. Таким чином, бізнесу однозначно варто залишити ворота саме на 30-му рівні для максимізації довгострокового утримання.")
else:
    print(f" - Наявність перетину свідчить про те, що діапазони можливих істинних значень утримання для обох версій перекриваються, тому для ухвалення остаточного рішення ми спираємося на результати точного Z-тесту.")

z statistic: 3.1644
p-value: 0.00155
Довірчий інтервал 95% для групи control: [0.1866, 0.1938]
Довірчий інтервал 95% для групи treatment: [0.1785, 0.1855]

 У якості висновку дайте відповідь на два питання:

1. Чи є статистична значущою різниця між поведінкою користувачів у різних версіях гри?
 Відповідь: Так, різниця між версіями гри є статистично значущою.
 Обґрунтування: Отримане значення p-value (0.00155) є значно меншим за заданий рівень значущості alpha (0.05). Це змушує нас відхилити нульову гіпотезу (H0)
 Практичний зміст: Перенесення перших лімітуючих воріт з 30-го на 40-й рівень викликало реальну статистично підтверджену зміну показника retention_7.

2. Чи перетинаються довірчі інтервали утримання користувачів з різних версій гри? Про що це каже?
 Відповідь: Ні, довірчі інтервали 95% для обох груп не перетинаються взагалі.
 Про що це каже:
 - Верхня межа інтервалу тестової групи treatment (0.1855) повністю знаходиться нижче, ніж нижня межа контрольної групи control (0.1866).


4. Виконайте тест Хі-квадрат на рівні значущості 5% аби визначити, чи є залежність між версією гри та утриманням гравця на 7ий день після реєстрації.

    - Напишіть, як для цього тесту будуть сформульовані гіпотези.
    - Проведіть обчислення, виведіть p-значення і напишіть висновок за результатами тесту.


In [40]:
from scipy.stats import chi2_contingency

contingency_table = pd.crosstab(df['version'], df['retention_7'])

chi2_stat, p_value, dof, expected = chi2_contingency(contingency_table)

print(f"\n1. ФОРМУЛЮВАННЯ СТАТИСТИЧНИХ ГІПОТЕЗ:")
print(f" - H0: Версія гри та утримання гравця на 7-й день є незалежними.")
print(f"   (Перенесення воріт не впливає на утримання).")
print(f" - Ha: Між версією гри та утриманням гравця на 7-й день є залежність.")
print(f"   (Перенесення воріт суттєво змінює показники утримання).")

print(f"\n2. ТАБЛИЦЯ СПРЯЖЕНОСТІ (ФАКТИЧНІ ДАНІ):")
print(contingency_table)

print(f"\n3. РЕЗУЛЬТАТИ ТЕСТУВАННЯ:")
print(f" - Статистика Хі-квадрат: {chi2_stat:.4f}")
print(f" - Значення p-value:       {p_value:.5f}")

print(f"\n4. ПРИЙНЯТТЯ РІШЕННЯ ЩОДО H0 (Рівень значущості alpha = 0.05):")
if p_value < 0.05:
    print(f" - Оскільки p-value ({p_value:.5f}) < alpha (0.05), ми відхиляємл нульову гіпотезу H0.")
else:
    print(f" - Оскільки p-value ({p_value:.5f}) >= alpha (0.05), ми НЕ відхиляємо нульову гіпотезу H0.")

print(f"\n5. ФІНАЛЬНИЙ ВИСНОВОК ЗА РЕЗУЛЬТАТАМИ ТЕСТУ:")
if p_value < 0.05:
    print(f" - Результати тесту Хі-квадрат підтверджують наявність статистично значущої залежності між версією гри (gate_30 чи gate_40) та утриманням гравців на 7-й день.")
    print(f"   Це повністю узгоджується з результатами попереднього Z-тесту: перенесення воріт на 40-й рівень погіршує показники, тому краще залишити їх на 30-му рівні.")
else:
    print(f" - Статистично значущої залежності між версією гри та утриманням на 7-й день не виявлено. Зміна рівня воріт не грає суттєвої ролі для довгострокового утримання.")


1. ФОРМУЛЮВАННЯ СТАТИСТИЧНИХ ГІПОТЕЗ:
 - H0: Версія гри та утримання гравця на 7-й день є незалежними.
   (Перенесення воріт не впливає на утримання).
 - Ha: Між версією гри та утриманням гравця на 7-й день є залежність.
   (Перенесення воріт суттєво змінює показники утримання).

2. ТАБЛИЦЯ СПРЯЖЕНОСТІ (ФАКТИЧНІ ДАНІ):
retention_7  False  True 
version                  
gate_30      36198   8502
gate_40      37210   8279

3. РЕЗУЛЬТАТИ ТЕСТУВАННЯ:
 - Статистика Хі-квадрат: 9.9591
 - Значення p-value:       0.00160

4. ПРИЙНЯТТЯ РІШЕННЯ ЩОДО H0 (Рівень значущості alpha = 0.05):
 - Оскільки p-value (0.00160) < alpha (0.05), ми відхиляємл нульову гіпотезу H0.

5. ФІНАЛЬНИЙ ВИСНОВОК ЗА РЕЗУЛЬТАТАМИ ТЕСТУ:
 - Результати тесту Хі-квадрат підтверджують наявність статистично значущої залежності між версією гри (gate_30 чи gate_40) та утриманням гравців на 7-й день.
   Це повністю узгоджується з результатами попереднього Z-тесту: перенесення воріт на 40-й рівень погіршує показники, тому краще 